# Phase 2B: Preprocessing & Sequence Design for LSTM Autoencoder

**Document:** Phase 2B Preprocessing Pipeline & Sequence Design  
**Date:** September 28, 2026  
**Baseline Dataset:** `df_train_normal` (Run ID: `21c851bc-384f-5b81-8747-6dcacdceff35` / `20260225_normal`, 11,991 rows)  

> **ANTI-LEAKAGE & SCOPE CERTIFICATION:**  
> 1. **Primary ML Domains:** Solar/PV and Wind only. Battery (BESS) and Grid Demand are strictly excluded from the ML feature matrix.  
> 2. **Baseline Only:** Fitted strictly on the uncompromised normal operational baseline. **ZERO** attack sessions or adversarial runs are used for fitting.  
> 3. **Chronological Splitting:** Validation data occurs strictly after training data in time. Zero random shuffling.  
> 4. **No-Lookahead Sequences:** Sliding window $X[t-L+1 : t]$ uses only causal past and current observations. Target $Y = X$.

In [ ]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.ml.preprocessing import (
    FEATURE_CONFIGS,
    CONSTANT_SIGNALS,
    SolarWindPreprocessor,
    prepare_preprocessed_datasets,
)
from src.ml.sequence_generator import (
    SequenceGenerator,
    compare_sequence_lengths,
    validate_all_leakage_assertions,
)

print('Phase 2B preprocessing and sequence modules imported successfully!')

## 1. Candidate Feature Configuration (Config A vs Config B)

We define two candidate feature sets based on the Phase 2A statistical findings:
- **CONFIG A (Broad Physical, 14 features):** Preserves all non-constant physical measurements and the PV contactor command. Crucially, it retains physical cross-sensor redundancies ($P_{ac}$ vs $P_{dc}$, dual anemometers, dual nacelle temps) so that the LSTM Autoencoder can detect single-sensor FDI spoofing by catching inconsistent cross-sensor relations.
- **CONFIG B (Reduced / Parsimonious, 10 features):** Removes collinear duplicate measurements ($P_{dc}$ where $r=1.000$, nacelle temp B where $r=0.9964$, anemometer B where $r=0.9769$) and excludes discrete control pulses ($pv_c_on_off$).
- **EXCLUDED SIGNALS:** `wind_m_height` (116m fixed), `wind_c_blade_rotation` (0.0°), and `wind_c_rotation_speed` (0.0 RPM) have zero variance and are excluded to avoid zero-division and dead neurons.

In [ ]:
print('=== CANDIDATE FEATURE CONFIGURATIONS ===')
for name, features in FEATURE_CONFIGS.items():
    print(f'\n{name.upper()} ({len(features)} features):')
    for i, feat in enumerate(features, 1):
        domain = 'Solar/PV' if feat.startswith('pv_') else 'Wind'
        sig_type = 'Control' if '_c_' in feat else 'Measurement'
        print(f'  {i:2d}. {feat:<26} [{domain} | {sig_type}]')

print('\n=== EXCLUDED CONSTANT SIGNALS ===')
for sig in CONSTANT_SIGNALS:
    print(f'  - {sig} (Zero variance in normal baseline)')

## 2. Chronological Train / Validation Partitioning

We partition the clean baseline dataset chronologically using an **80/20 train/validation split**:
- Training: First 80% (9,592 samples, ~1.40 hours)
- Validation: Subsequent 20% (2,399 samples, ~0.35 hours)
- Zero random shuffling to strictly preserve temporal ordering.

In [ ]:
data_a = prepare_preprocessed_datasets(config_name='config_a', scaler_type='minmax')
split_info = data_a['split_info']

print('=== CHRONOLOGICAL SPLIT DETAILS ===')
print(f"Total Baseline Rows:     {split_info['total_samples']:,}")
print(f"Training Rows (80%):     {split_info['train_samples']:,}")
print(f"Validation Rows (20%):   {split_info['val_samples']:,}")
print(f"Training Time Window:    {split_info['train_start_ts']} to {split_info['train_end_ts']}")
print(f"Validation Time Window:  {split_info['val_start_ts']} to {split_info['val_end_ts']}")
print(f"Boundary Temporal Gap:   {split_info['temporal_gap_at_boundary_sec']:.4f} seconds")

# Verify strict forward chronology
assert split_info['val_start_ts'] > split_info['train_end_ts'], 'Chronological boundary violation!'

### Train / Validation Split Timeline Visualization
![Train/Val Timeline](../reports/figures/train_val_split_timeline.png)

## 3. Scaling Strategy Evaluation: MinMaxScaler(-1, 1) vs RobustScaler

- **MinMaxScaler with range [-1, 1]:** Linearly maps features strictly into $[-1.0, 1.0]$. Provides a bounded and consistent input range, making heterogeneous physical measurements numerically comparable and providing an input range compatible with the LSTM nonlinearities. **Recommended initial baseline.**
- **RobustScaler:** Centers by median and scales by IQR ($Q_3 - Q_1$). Because physical signals like solar irradiance have positive skew and long tails, robust scaling causes extreme tail values to map well beyond $[-1, 1]$ (up to $+3.64$).

In [ ]:
data_mm = prepare_preprocessed_datasets(config_name='config_a', scaler_type='minmax')
data_rb = prepare_preprocessed_datasets(config_name='config_a', scaler_type='robust')

print('=== SCALED VALUE RANGE COMPARISON ===')
print(f"MinMaxScaler Train Range: [{data_mm['train_scaled'].min():.4f}, {data_mm['train_scaled'].max():.4f}]")
print(f"MinMaxScaler Val Range:   [{data_mm['val_scaled'].min():.4f}, {data_mm['val_scaled'].max():.4f}]")
print(f"RobustScaler Train Range: [{data_rb['train_scaled'].min():.4f}, {data_rb['train_scaled'].max():.4f}]")
print(f"RobustScaler Val Range:   [{data_rb['val_scaled'].min():.4f}, {data_rb['val_scaled'].max():.4f}]")

### Scaling Distribution Comparison Visualization
![Scaling Comparison](../reports/figures/scaling_comparison.png)

## 4. Sliding-Window Sequence Generation

Sequences are generated using a sliding window $X[t-L+1 : t]$ with reconstruction target $Y = X$.
We evaluate three candidate sequence lengths given the nominal sampling interval of $\Delta t \approx 0.5262$ seconds:
- **L = 30:** $\approx 15.8$ seconds of physical context
- **L = 60:** $\approx 31.6$ seconds of physical context (**Recommended Baseline** — provides a reasonable balance between temporal context, sequence length, and computational cost)
- **L = 120:** $\approx 63.1$ seconds of physical context

In [ ]:
cmp_table = compare_sequence_lengths(data_mm)
display(cmp_table[[
    'sequence_length_L',
    'approx_duration_formatted',
    'train_sequences',
    'val_sequences',
    'total_sequences',
    'input_shape_train',
    'input_shape_val',
    'strided_memory_mb'
]])

### Sequence Length Comparison Visualization
![Sequence Length Comparison](../reports/figures/sequence_length_comparison.png)

## 5. Executable Data Leakage Assertions

We execute all 9 mandatory anti-leakage verification checks across the end-to-end preprocessing and sequence generation pipeline.

In [ ]:
gen = SequenceGenerator(sequence_length=60)
seq_data = gen.generate_train_val_sequences(data_mm)
checks = validate_all_leakage_assertions(data_mm, seq_data)

check_rows = []
for check_name, (passed, desc) in checks.items():
    check_rows.append({
        'Assertion Check': check_name,
        'Status': 'PASS' if passed else 'FAIL',
        'Validation Details': desc
    })

checks_df = pd.DataFrame(check_rows)
display(checks_df.style.applymap(lambda v: 'color: green; font-weight: bold;' if v == 'PASS' else 'color: red; font-weight: bold;', subset=['Status']))

assert all(p for p, _ in checks.values()), 'One or more leakage assertions failed!'

## 6. Recommended Baseline Configuration for Phase 2C

| Parameter | Recommended Specification | Rationale |
| :--- | :--- | :--- |
| **Feature Configuration** | **CONFIG A (14 features)** | Preserves physical cross-sensor correlations ($P_{ac}$ vs $P_{dc}$, anemometer A/B, nacelle temp A/B) essential for detecting FDI attacks. |
| **Scaler** | **MinMaxScaler(range=(-1, 1))** | Bounded and consistent input range, making heterogeneous physical measurements numerically comparable and compatible with LSTM nonlinearities. |
| **Split Strategy** | **Chronological 80 / 20** | Train: 9,592 rows; Val: 2,399 rows; 0.525s positive temporal boundary gap. |
| **Sequence Length ($L$)** | **$L = 60$ (31.6 seconds)** | Provides approximately 31.6s of temporal context, balancing context depth and computational cost (to be validated empirically in Phase 2C). |
| **Input Shape** | **Train: `(9533, 60, 14)` / Val: `(2340, 60, 14)`** | Target $Y = X$ for autoencoder reconstruction. |
| **Memory Strategy** | **NumPy Strided Views** | Shared memory array (~1 MB), avoiding unnecessary 64 MB duplications. |